In [ ]:
from pathlib import Path
import re
import time
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

warnings.filterwarnings("ignore")

In [ ]:
from pathlib import Path
import os

def _find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "README.md").exists() and (candidate / "notebooks").is_dir():
            return candidate
    return current

PROJECT_ROOT = _find_project_root()
DATA_ROOT = Path(os.environ.get("GANGA_DATA_ROOT", PROJECT_ROOT / "data")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("GANGA_OUTPUT_ROOT", PROJECT_ROOT / "outputs")).expanduser()

PROJECT_PATH = PROJECT_ROOT
# This folder is expected to contain the monthly/weekly optical TIFFs used by this notebook.
LANDSAT_FOLDER = Path(os.environ.get("GANGA_OPTICAL_RASTER_DIR", DATA_ROOT / "satellite" / "optical_rasters")).expanduser()
VIIRS_FOLDER = Path(os.environ.get("GANGA_VIIRS_DIR", DATA_ROOT / "satellite" / "VIIRS_2015_2025_COLOR")).expanduser()
LULC_FOLDER = Path(os.environ.get("GANGA_LULC_DIR", DATA_ROOT / "satellite" / "LULC_MONTHLY_2015_2025")).expanduser()
STATION_FILE = OUTPUT_ROOT / "monitoring_stations" / "Monitoring_Stations.shp"
WEEKLY_CSV = DATA_ROOT / "metadata" / "weekly_best_dates.csv"
FEATURE_FOLDER = OUTPUT_ROOT / "features"
FEATURE_FOLDER.mkdir(parents=True, exist_ok=True)

print(f"Optical rasters: {LANDSAT_FOLDER}")
print(f"VIIRS rasters: {VIIRS_FOLDER}")
print(f"LULC rasters: {LULC_FOLDER}")
print(f"Feature output: {FEATURE_FOLDER}")


In [ ]:
stations = gpd.read_file(STATION_FILE)

stations = stations.to_crs(4326)

coords = [(p.x, p.y) for p in stations.geometry]

print(len(coords))

finding monthly tiffs.. 2015-2018

In [ ]:
monthly_files = sorted([
    f for f in LANDSAT_FOLDER.glob("*.tif")
    if re.match(r"^\d{2}_\d{4}\.tif$", f.name)
])

print(f"Found {len(monthly_files)} monthly files")

for f in monthly_files[:5]:
    print(f.name)

RGB Extraction Function

In [ ]:
def extract_rgb(raster):

    for attempt in range(3):

        try:

            with rasterio.open(raster) as src:

                values = np.array(list(src.sample(coords)))

            return values

        except Exception:

            print(
                f"Retry {attempt+1} : {raster.name}"
            )

            time.sleep(2)

    print(
        f"Skipped : {raster.name}"
    )

    return None

In [ ]:
monthly_results = []

for raster in monthly_files:

    month = int(raster.stem.split("_")[0])
    year = int(raster.stem.split("_")[1])

    print(f"Processing {year}-{month:02d}")

    rgb = extract_rgb(raster)

    if rgb is None:
        continue

    df = pd.DataFrame(
        rgb,
        columns=["Blue", "Green", "Red"]
    )

    df.insert(
        0,
        "Station_ID",
        stations["Station_ID"].values
    )

    df["Year"] = year
    df["Month"] = month

    monthly_results.append(df)

In [ ]:
landsat_features = pd.concat(
    monthly_results,
    ignore_index=True
)

landsat_features = landsat_features.sort_values(
    ["Year", "Month", "Station_ID"]
).reset_index(drop=True)

print(landsat_features.shape)

landsat_features.head()

In [ ]:
landsat_features.to_csv(
    FEATURE_FOLDER / "Monthly_RGB_2015_2018.csv",
    index=False
)

print("✅ Monthly RGB saved successfully!")

In [ ]:
weekly_df = pd.read_csv(WEEKLY_CSV)

weekly_df["best_image_date"] = pd.to_datetime(
    weekly_df["best_image_date"]
)

weekly_df["Year"] = weekly_df["best_image_date"].dt.year

weekly_df["Month"] = weekly_df["best_image_date"].dt.month

weekly_df = weekly_df[
    weekly_df["Year"] >= 2019
]

In [ ]:
#week finder
def find_week_raster(week):

    patterns = [

        f"WEEK_{week}.tif",

        f"WEEK {week}.tif",

        f"week_{week}.tif",

        f"week {week}.tif"

    ]

    for p in patterns:

        f = LANDSAT_FOLDER / p

        if f.exists():

            return f

    return None

In [ ]:
#save checkpoint
def save_year(df, year):

    file = FEATURE_FOLDER / f"RGB_{year}.csv"

    df.to_csv(file, index=False)

    print(f"Saved {file.name}")

In [ ]:
#year processor
def process_year(year):

    outfile = FEATURE_FOLDER / f"RGB_{year}.csv"

    if outfile.exists():

        print(f"{year} already done")

        return

    year_df = weekly_df[
        weekly_df["Year"] == year
    ]

    result = []

    groups = year_df.groupby(["Year","Month"])

    for (_, month), group in groups:

        print(f"\n{year}-{month:02d}")

        monthly_rgb = []

        for week in group["week"]:

            raster = find_week_raster(
                int(week)
            )

            if raster is None:

                print(f"Week {week} Missing")

                continue

            rgb = extract_rgb(raster)

            if rgb is None:

                continue

            monthly_rgb.append(rgb)

        if len(monthly_rgb)==0:

            continue

        monthly_rgb = np.nanmean(
            np.stack(monthly_rgb),
            axis=0
        )

        df = pd.DataFrame(

            monthly_rgb,

            columns=[
                "Blue",
                "Green",
                "Red"
            ]

        )

        df.insert(
            0,
            "Station_ID",
            stations["Station_ID"]
        )

        df["Year"] = year

        df["Month"] = month

        result.append(df)

    result = pd.concat(
        result,
        ignore_index=True
    )

    save_year(
        result,
        year
    )

In [ ]:
for year in range(2019,2026):

    process_year(year)

In [ ]:
files = sorted(FEATURE_FOLDER.glob("RGB_20*.csv"))

weekly_rgb = pd.concat(
    [pd.read_csv(f) for f in files],
    ignore_index=True
)

print(weekly_rgb.shape)

In [ ]:
#merge both 2015-2018 and 2019-2025
landsat_features = pd.concat(
    [landsat_features, weekly_rgb],
    ignore_index=True
)

landsat_features = landsat_features.sort_values(
    ["Year", "Month", "Station_ID"]
).reset_index(drop=True)

print(landsat_features.shape)

In [ ]:
landsat_features.to_csv(
    FEATURE_FOLDER / "Landsat_Features_2015_2025.csv",
    index=False
)

print("🎉 Landsat Feature Extraction Completed Successfully!")

LANDSAT DONE CSV OBTAINED
NOW VIIRS CSV REQUIRED


In [ ]:
from pathlib import Path

viirs_files = sorted(VIIRS_FOLDER.glob("*.tif"))

print("Total files:", len(viirs_files))

for f in viirs_files[:15]:
    print(f.name)

BUILDING VIIRS INVENTORY

In [ ]:
import re

viirs_files = sorted(VIIRS_FOLDER.glob("*.tif"))

records = []

pattern = re.compile(r"VIIRS_(\d{4})_(\d{2})_(\d{2})\.tif")

for f in viirs_files:

    m = pattern.match(f.name)

    if m:

        year = int(m.group(1))
        month = int(m.group(2))
        day = int(m.group(3))

        records.append({
            "Year": year,
            "Month": month,
            "Day": day,
            "Path": f
        })

viirs_df = pd.DataFrame(records)

print(viirs_df.head())
print()
print("Total Images:", len(viirs_df))

VIIRS EXTRACTION FEATURE

In [ ]:
def extract_viirs(raster):

    for attempt in range(3):

        try:

            with rasterio.open(raster) as src:

                values = np.array(list(src.sample(coords)))

            return values[:, 0]

        except Exception:

            print(f"Retry {attempt+1}: {raster.name}")

            time.sleep(2)

    print(f"Skipped: {raster.name}")

    return None

MONTHLY PROCESSING

In [ ]:
viirs_results = []

groups = viirs_df.groupby(["Year", "Month"])

for (year, month), group in groups:

    print(f"Processing {year}-{month:02d}")

    monthly = []

    for raster in group["Path"]:

        values = extract_viirs(raster)

        if values is not None:
            monthly.append(values)

    if len(monthly) == 0:
        continue

    monthly = np.nanmean(np.stack(monthly), axis=0)

    df = pd.DataFrame({
        "Station_ID": stations["Station_ID"].values,
        "Year": year,
        "Month": month,
        "VIIRS": monthly
    })

    viirs_results.append(df)

In [ ]:
viirs_features = pd.concat(
    viirs_results,
    ignore_index=True
)

viirs_features = viirs_features.sort_values(
    ["Year", "Month", "Station_ID"]
).reset_index(drop=True)

print(viirs_features.shape)

viirs_features.to_csv(
    FEATURE_FOLDER / "VIIRS_Features_2015_2025.csv",
    index=False
)

print("✅ VIIRS feature extraction completed")

In [ ]:
landsat = pd.read_csv(
    FEATURE_FOLDER / "Landsat_Features_2015_2025.csv"
)

viirs = pd.read_csv(
    FEATURE_FOLDER / "VIIRS_Features_2015_2025.csv"
)

print("Landsat :", landsat.shape)
print("VIIRS   :", viirs.shape)

print("\nMissing Landsat:", landsat.isna().sum().sum())
print("Missing VIIRS  :", viirs.isna().sum().sum())

In [ ]:
test = landsat.merge(
    viirs,
    on=["Station_ID", "Year", "Month"],
    how="inner"
)

print(test.shape)
test.head()

In [ ]:
landsat.isna().sum()

In [ ]:
missing = landsat[landsat[["Blue", "Green", "Red"]].isna().any(axis=1)]

print("Missing rows:", len(missing))

missing.head()

In [ ]:
missing.groupby(["Year", "Month"]).size().sort_values(ascending=False)

In [ ]:
missing["Station_ID"].nunique()

LULC STARTING

In [ ]:
lulc_files = sorted(LULC_FOLDER.glob("*.tif"))

print("Total files:", len(lulc_files))

for f in lulc_files[:20]:
    print(f.name)

In [ ]:
import re

lulc_files = sorted(LULC_FOLDER.glob("*.tif"))

records = []

pattern = re.compile(r"Ganga_LULC_(\d{4})_(\d{2})\.tif")

for f in lulc_files:

    m = pattern.match(f.name)

    if m:

        records.append({

            "Year": int(m.group(1)),
            "Month": int(m.group(2)),
            "Path": f

        })

lulc_df = pd.DataFrame(records)

print(lulc_df.head())

print()

print("Total LULC Files :", len(lulc_df))

LULC EXTRACTION FEATURE


In [ ]:
def extract_lulc(raster):

    for attempt in range(3):

        try:

            with rasterio.open(raster) as src:

                values = np.array(list(src.sample(coords)))

            return values[:,0]

        except Exception:

            print(f"Retry {attempt+1}: {raster.name}")

            time.sleep(2)

    print(f"Skipped : {raster.name}")

    return None

LULC PROCESSING

In [ ]:
lulc_results = []

for _, row in lulc_df.iterrows():

    year = row["Year"]
    month = row["Month"]

    print(f"Processing {year}-{month:02d}")

    values = extract_lulc(row["Path"])

    if values is None:
        continue

    df = pd.DataFrame({

        "Station_ID": stations["Station_ID"].values,

        "Year": year,

        "Month": month,

        "LULC": values

    })

    lulc_results.append(df)

In [ ]:
lulc_features = pd.concat(
    lulc_results,
    ignore_index=True
)

lulc_features = lulc_features.sort_values(
    ["Year","Month","Station_ID"]
).reset_index(drop=True)

print(lulc_features.shape)

lulc_features.head()

In [ ]:
lulc_features.to_csv(
    FEATURE_FOLDER / "LULC_Features_2015_2025.csv",
    index=False
)

print("✅ LULC Feature Extraction Completed")

In [ ]:
print(lulc_features.shape)

print()

print(lulc_features["LULC"].value_counts().sort_index())

merging all csv


In [ ]:
landsat = pd.read_csv(
    FEATURE_FOLDER / "Landsat_Features_2015_2025.csv"
)

viirs = pd.read_csv(
    FEATURE_FOLDER / "VIIRS_Features_2015_2025.csv"
)

lulc = pd.read_csv(
    FEATURE_FOLDER / "LULC_Features_2015_2025.csv"
)

print("Landsat :", landsat.shape)
print("VIIRS   :", viirs.shape)
print("LULC    :", lulc.shape)

In [ ]:
master = landsat.merge(

    viirs,

    on=["Station_ID","Year","Month"],

    how="inner"

)

print(master.shape)

merge

In [ ]:
master = master.merge(

    lulc,

    on=["Station_ID","Year","Month"],

    how="inner"

)

print(master.shape)
master.head()

In [ ]:
print("="*50)

print("Rows :", len(master))

print("Columns :", len(master.columns))

print()

print(master.isna().sum())

print()

print("Duplicate Rows :", master.duplicated().sum())

print()

print("Unique Stations :", master["Station_ID"].nunique())

print()

print("Years :", sorted(master["Year"].unique()))

print()

print("Months :", sorted(master["Month"].unique()))

In [ ]:
master.info()

In [ ]:
master.to_csv(

    FEATURE_FOLDER / "Master_Features_2015_2025.csv",

    index=False

)

print("🎉 Master Feature Dataset Saved Successfully!")

In [ ]:
master.sample(10)